### 1. Imports

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim

### 2. Building the perceptron

nn.Linear creates a fully connected neural network layer. It also uses **LeCun initialization** technique, so the values are in the range (-0.7071, 0.7071). Since I want to replicated the Perceptron as close as possible, I am overriding those values with **random uniform initialization** in range (-0.1, 0.1)

### Different Initialization techniques:

* Zero
###### # every weight is 0
nn.init.zeros_(layer.weight)
nn.init.zeros_(layer.bias)

* Random uniform
###### # arbitrary small range
nn.init.uniform_(layer.weights, -0.1, 0.1)
nn.init.uniform_(layer.bias, -0.1, 0.1)

* He / Kaiming
###### # std = sqrt(2/fan_in)
nn.init.kaiming_normal_(layer.weight, mode='fan_in', nonlinearity='relu')

* Xavier
##### # std = sqrt(1/fan_avg)
nn.linear.xavier_uniform_(layer.weight)

* LeCun
##### # std = sqrt(1/fan_in)
nn.init.kaiming_normal_(layer.weights, mode='fan_in', nonlinearity='linear')

Rule of thumb:
* ReLU -> He
* tanh/sigmid -> Xavier
* No activation/linear -> LeCun
* Zero -> barely used (mostly fot bias initialization)

`forward` — the prediction

`forward` defines what happens when you call `model(x)`. It returns the raw **logit**:

```
logit = w1*x1 + w2*x2 + bias
```

https://www.lightly.ai/blog/pytorch-loss-functions$0
Here is a website that contains a table summarizing all the loss functions and when they are most suitable.

In [36]:
class Perceptron(nn.Module):
  def __init__(self):
    super().__init__()
    self.linear = nn.Linear(2, 1)
    nn.init.uniform_(self.linear.weight, -0.1, 0.1)
    nn.init.uniform_(self.linear.bias,   -0.1, 0.1)

  def show_weights(self) -> None:
        w1, w2 = self.linear.weight.data[0]
        b       = self.linear.bias.data[0]
        print(f"{w1:+.4f} {w2:+.4f} {b:+.4f}")

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.linear(x)

  def train_model(self, X, y, max_epochs= 10000):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.SGD(self.parameters(), lr=0.1)
    for epoch in range(1, max_epochs + 1):
            self.train() # train mode
            optimizer.zero_grad() # clear gradients from previous step
            loss = criterion(self(X), y.unsqueeze(1)) # compute loss
            loss.backward() # compute gradients
            optimizer.step() # update weights
            with torch.no_grad():
                predictions = (self(X) >= 0).float().squeeze()
                tss = ((y - predictions) ** 2).sum().item()
            if tss == 0:
                print(f"All patterns learned  (epoch {epoch},  loss {loss.item():.4f})")
                return
    print(f"Did not converge within {max_epochs} epochs.")

  def evaluate(self, patterns, targets):
        self.eval()  # evaluation mode
        with torch.no_grad():
            predictions = (self(X) >= 0).float().squeeze()

        for pattern, target, output in zip(patterns, targets, predictions.tolist()):
            if int(output) == target:
                print(f"{pattern} --> {int(output)}")
            else:
                print(f"{pattern} --> {int(output)}  (WRONG, should be {target})")

        tss_error = ((targets - predictions) ** 2).sum().item()
        print(f"TSS error = {tss_error:.1f}")

### 3. Preparing the data

In [37]:
# 2-bit input patterns
patterns = [[0,0], [0,1], [1,0], [1,1]]

# targets for 2-bit input patterns
ANDtargets  = [0, 0, 0, 1]
ORtargets   = [0, 1, 1, 1]
NANDtargets = [1, 1, 1, 0]
NORtargets  = [1, 0, 0, 0]
XORtargets  = [0, 1, 1, 0]

In [38]:
X           = torch.tensor(patterns,    dtype=torch.float32)
ANDtargets  = torch.tensor(ANDtargets,  dtype=torch.float32)
ORtargets   = torch.tensor(ORtargets,   dtype=torch.float32)
NANDtargets = torch.tensor(NANDtargets, dtype=torch.float32)
NORtargets  = torch.tensor(NORtargets,  dtype=torch.float32)
XORtargets  = torch.tensor(XORtargets,  dtype=torch.float32)

### 4. Try it out!

In [39]:
model = Perceptron()
print("Initial weight:", model.linear.weight.data)
print("Initial bias:  ", model.linear.bias.data)

Initial weight: tensor([[-0.0560,  0.0313]])
Initial bias:   tensor([-0.0732])


In [47]:
model = Perceptron()
model.train_model(X, ORtargets, max_epochs=10000)
model.evaluate(patterns, ORtargets)
w1, w2 = model.linear.weight.data[0]
b = model.linear.bias.data[0]
print(f"Final weights: w1={w1:+.4f}  w2={w2:+.4f}  bias={b:+.4f}")

All patterns learned  (epoch 193,  loss 0.2716)
[0, 0] --> 0
[0, 1] --> 1
[1, 0] --> 1
[1, 1] --> 1
TSS error = 0.0
Final weights: w1=+1.6496  w2=+1.6326  bias=-0.0009
